# Detección de anomalías en videos de vigilancia (TensorFlow)

El modelo indica **si un video contiene una anomalía y en qué segmento ocurre**.

1. Edita la celda **1.1** (rutas y modo `"train"` o `"load"`).
2. Ejecuta todo (*Run All*): entrena o carga el modelo, guarda los pesos y muestra las métricas.
3. En la última celda elige tu video y el modelo para ver el video de salida.

Entrenamiento: features CLIP precalculadas de VadCLIP (UCF-Crime) → Transformer temporal en Keras → puntuación de anomalía por fragmento de 16 frames.

Requisitos: `pip install "tensorflow[and-cuda]" open_clip_torch opencv-python pandas scikit-learn matplotlib pillow` y `ffmpeg`. En Colab basta con `!pip install open_clip_torch`.

## 1. Configuración

In [10]:
# 1.1 User parameters
ENVIRONMENT = "local"            # "local" | "colab"
MODE = "train"                   # "train": train and save a new model · "load": load saved weights

BASE = "."
LOCAL_PATHS = {
    "videos":   f"{BASE}/archive/UCF_Crimes/UCF_Crimes/Videos",   # UCF-Crime .mp4 (only needed for the demo)
    "features": f"{BASE}/UCFClipFeatures",                        # VadCLIP .npy features
    "lists":    f"{BASE}/VadCLIP-main/list",                      # official lists (empty = clone from GitHub)
    "work":     f"{BASE}/trabajo",                                # outputs
}
COLAB_PATHS = {"videos": "/content/drive/MyDrive/UCF-Crime", "features": "/content/drive/MyDrive/UCFClipFeatures",
               "lists": "", "work": "/content/drive/MyDrive/vad_trabajo"}

MODEL_NAME = "modelo_tf"         # weights saved as <work>/modelo_tf/<MODEL_NAME>.weights.h5

In [11]:
# 1.2 Libraries, GPU and seeds
import os, re, json, random, subprocess
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd
import matplotlib, matplotlib.pyplot as plt
import cv2
import torch, open_clip                 # import before TensorFlow (avoids a native-library clash); only used for CLIP in 5.1
os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")   # less GPU memory fragmentation
import tensorflow as tf, keras
from keras import layers
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, Video
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)        # leave GPU memory for CLIP in the demo
SEED = 42
random.seed(SEED); np.random.seed(SEED); keras.utils.set_random_seed(SEED)
print("TensorFlow", tf.__version__, "· GPU:", [g.name for g in tf.config.list_physical_devices("GPU")] or "none")

if ENVIRONMENT == "colab":
    from google.colab import drive; drive.mount("/content/drive")
PATHS = COLAB_PATHS if ENVIRONMENT == "colab" else LOCAL_PATHS
WORK = Path(PATHS["work"]).resolve() / "modelo_tf"
for sub in ["", "figuras", "demo", "features_propias"]:
    (WORK / sub).mkdir(parents=True, exist_ok=True)
WEIGHTS_PATH = WORK / f"{MODEL_NAME}.weights.h5"
CONFIG_PATH = WORK / f"{MODEL_NAME}.json"
print("Outputs:", WORK)

TensorFlow 2.22.0-rc0 · GPU: ['/physical_device:GPU:0']
Outputs: /mnt/af724231-956c-44f8-ae04-79cc6b3e18d2/IA_2_Dataset/trabajo/modelo_tf


In [12]:
# 1.3 Hyperparameters
FRAG = 16                        # frames per fragment (one CLIP feature per fragment)
FPS = 30.0                       # UCF-Crime frame rate
MAX_LEN = 256                    # fragments per training sequence
DIM, LAYERS, HEADS, FFN = 512, 2, 8, 2048
EPOCHS, BATCH, LR, WEIGHT_DECAY = 10, 64, 2e-5, 1e-2
MICRO_BATCH = 16                 # videos per forward pass; gradients are accumulated up to BATCH (lower it if the GPU runs out of memory)
VAL_FRAC = 0.15

# loss weights
W_BOTTOMK, BOTTOMK_FRAC = 0.5, 0.3   # lowest 30 % of an anomalous video should look normal
W_SPARSE, W_SMOOTH = 0.2, 0.1        # few high fragments; smooth over time

# post-processing (fragment scores -> anomalous intervals)
EMA_ALPHA, OFF_RATIO = 0.5, 0.8
MIN_EVENT_SEC, MERGE_GAP_SEC = 2.0, 5.0
IOU_MATCH = 0.3                      # an interval counts as detected if IoU >= 0.3

## 2. Datos

In [13]:
# 2.1 Official split, test ground truth and feature index
def list_dir():
    found = sorted(Path(PATHS["lists"]).rglob("Temporal_Anomaly_Annotation.txt")) if PATHS["lists"] else []
    if not found:
        repo = WORK.parent / "repos/VadCLIP"
        if not repo.exists():
            subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/nwpu-zxr/VadCLIP.git", str(repo)], check=True)
        found = sorted(repo.rglob("Temporal_Anomaly_Annotation.txt"))
    return found[0].parent

LISTS = list_dir()
read_list = lambda f: [Path(l.strip()).stem for l in open(LISTS / f) if l.strip()]
TRAIN_NAMES, TEST_NAMES = read_list("Anomaly_Train.txt"), read_list("Anomaly_Test.txt")

GT = {}                          # test video -> anomalous intervals in frames
for l in open(LISTS / "Temporal_Anomaly_Annotation.txt"):
    p = l.split()
    if len(p) >= 6:
        GT[Path(p[0]).stem] = [(int(p[i]), int(p[i + 1])) for i in (2, 4) if int(p[i]) >= 0]

FEATS = defaultdict(dict)        # video -> {crop: .npy path}
for f in Path(PATHS["features"]).rglob("*.npy"):
    m = re.match(r"(.+?)__(\d+)$", f.stem)
    if m:
        FEATS[m.group(1)][int(m.group(2))] = str(f)
VIDEOS = {f.stem: str(f) for ext in ("mp4", "avi", "mkv", "mov") for f in Path(PATHS["videos"]).rglob(f"*.{ext}")} \
         if Path(PATHS["videos"]).exists() else {}

table = pd.DataFrame({"video": list(dict.fromkeys(TRAIN_NAMES + TEST_NAMES))})
table["split"] = np.where(table.video.isin(set(TEST_NAMES)), "test", "train")
table["clase"] = [("Normal" if v.startswith("Normal") else re.match(r"^([A-Za-z]+?)\d", v).group(1)) for v in table.video]
table["anomalo"] = (table.clase != "Normal").astype(int)
table = table[table.video.isin(FEATS)].reset_index(drop=True)
_tr = table.index[table.split == "train"]
_, _val = train_test_split(_tr, test_size=VAL_FRAC, stratify=table.loc[_tr, "clase"], random_state=SEED)
table.loc[_val, "split"] = "val"
display(pd.crosstab(table.clase, table.split, margins=True))
print(f"Features for {len(FEATS)} videos · {len(VIDEOS)} video files found")

split,test,train,val,All
clase,,,,
Abuse,2,41,7,50
Arrest,5,38,7,50
Arson,9,35,6,50
Assault,3,40,7,50
Burglary,13,74,13,100
Explosion,21,24,5,50
Fighting,5,38,7,50
Normal,150,680,120,950
RoadAccidents,23,108,19,150


Features for 1900 videos · 1900 video files found


In [14]:
# 2.2 Feature loading and batch generator
def load_features(video, crop=None):
    crops = FEATS[video]
    crop = crop if crop in crops else (0 if 0 in crops else min(crops))
    return np.load(crops[crop]).astype(np.float32)

def fit_length(x, L=MAX_LEN):
    """Pad short videos; compress long ones by averaging consecutive fragments."""
    T = len(x)
    if T > L:
        r = np.linspace(0, T, L + 1).astype(int)
        return np.stack([x[r[i]:max(r[i + 1], r[i] + 1)].mean(0) for i in range(L)]), L
    return np.pad(x, ((0, L - T), (0, 0))), T

TRAIN = table[table.split == "train"]
NORMAL, ANOMAL = TRAIN.video[TRAIN.anomalo == 0].tolist(), TRAIN.video[TRAIN.anomalo == 1].tolist()
STEPS_PER_EPOCH = len(TRAIN) * 10 // BATCH        # ~ every crop of every video once per epoch

def batches(B=MICRO_BATCH):
    """Balanced micro-batches (half normal, half anomalous) with a random VadCLIP crop per video."""
    while True:
        vids = random.choices(NORMAL, k=B // 2) + random.choices(ANOMAL, k=B - B // 2)
        xs, ls = zip(*(fit_length(load_features(v, random.choice(list(FEATS[v])))) for v in vids))
        yield np.stack(xs), np.array(ls, np.int32), np.array([0] * (B // 2) + [1] * (B - B // 2), np.float32)

train_ds = tf.data.Dataset.from_generator(batches, output_signature=(
    tf.TensorSpec((MICRO_BATCH, MAX_LEN, DIM), tf.float32), tf.TensorSpec((MICRO_BATCH,), tf.int32),
    tf.TensorSpec((MICRO_BATCH,), tf.float32))).prefetch(2)

## 3. Modelo (Keras)

In [15]:
# 3.1 Temporal model: Transformer encoder + anomaly head
@keras.saving.register_keras_serializable()
class AnomalyModel(keras.Model):
    def __init__(self, **kw):
        super().__init__(**kw)
        self.inp_norm, self.inp = layers.LayerNormalization(), layers.Dense(DIM)
        self.pos = self.add_weight(shape=(1, MAX_LEN, DIM), initializer=keras.initializers.TruncatedNormal(stddev=0.02), name="pos")
        self.blocks = [dict(n1=layers.LayerNormalization(), att=layers.MultiHeadAttention(HEADS, DIM // HEADS, dropout=0.1),
                            n2=layers.LayerNormalization(), ff1=layers.Dense(FFN, activation="gelu"), ff2=layers.Dense(DIM),
                            drop=layers.Dropout(0.1)) for _ in range(LAYERS)]
        self.out_norm, self.head = layers.LayerNormalization(), layers.Dense(1)

    def call(self, x, lengths, training=False):
        T = tf.shape(x)[1]
        valid = tf.sequence_mask(lengths, T)                                   # (B, T) real fragments
        att_mask = valid[:, None, :] & tf.ones_like(valid)[:, :, None]          # attend only to real fragments
        h = self.inp(self.inp_norm(x)) + self.pos[:, :T]
        for b in self.blocks:                                                   # pre-norm Transformer block
            h = h + b["drop"](b["att"](b["n1"](h), b["n1"](h), attention_mask=att_mask, training=training), training=training)
            h = h + b["drop"](b["ff2"](b["ff1"](b["n2"](h))), training=training)
        return tf.squeeze(tf.sigmoid(self.head(self.out_norm(h))), -1)        # (B, T) anomaly score

def build_model():
    m = AnomalyModel()
    m(tf.zeros((1, MAX_LEN, DIM)), tf.constant([MAX_LEN]))                      # create the weights
    return m

def predict(model, x):
    """Anomaly score for every fragment of a full video (processed in chunks of MAX_LEN)."""
    out = []
    for i in range(0, len(x), MAX_LEN):
        xi = x[i:i + MAX_LEN]; Li = len(xi)
        s = model(np.pad(xi, ((0, MAX_LEN - Li), (0, 0)))[None], tf.constant([Li]))
        out.append(s.numpy()[0, :Li])
    return np.concatenate(out)

def video_score(s):
    k = len(s) // 16 + 1
    return float(np.sort(s)[-k:].mean())

In [16]:
# 3.2 Losses: MIL top-k + bottom-k + sparsity + smoothness
def masked_sorted(s, valid, descending):
    fill = -1.0 if descending else 2.0
    return tf.sort(tf.where(valid, s, fill), axis=1, direction="DESCENDING" if descending else "ASCENDING")

def compute_loss(s, lengths, y):
    T = tf.shape(s)[1]
    valid = tf.sequence_mask(lengths, T)
    L = tf.cast(lengths, tf.float32)
    rank = tf.cast(tf.range(T)[None], tf.float32)
    eps = 1e-6
    # MIL top-k: mean of the k highest scores vs the video label
    k = tf.floor(L / 16) + 1
    top = tf.reduce_sum(masked_sorted(s, valid, True) * tf.cast(rank < k[:, None], tf.float32), 1) / k
    top = tf.clip_by_value(top, eps, 1 - eps)
    l_mil = -tf.reduce_mean(y * tf.math.log(top) + (1 - y) * tf.math.log(1 - top))
    # bottom-k: the lowest scores of anomalous videos should be ~0
    kb = tf.maximum(1.0, tf.floor(BOTTOMK_FRAC * L))
    low = masked_sorted(s, valid, False)
    l_bot = tf.reduce_sum(-tf.math.log(1 - tf.clip_by_value(low, 0, 1 - eps)) * tf.cast(rank < kb[:, None], tf.float32), 1) / kb
    l_bot = tf.reduce_sum(l_bot * y) / tf.maximum(tf.reduce_sum(y), 1)
    # sparsity: anomalous videos should have few high fragments
    mean_s = tf.reduce_sum(s * tf.cast(valid, tf.float32), 1) / L
    l_sparse = tf.reduce_sum(mean_s * y) / tf.maximum(tf.reduce_sum(y), 1)
    # smoothness: scores change gradually
    pair = tf.cast(valid[:, 1:] & valid[:, :-1], tf.float32)
    l_smooth = tf.reduce_sum(tf.square(s[:, 1:] - s[:, :-1]) * pair) / tf.maximum(tf.reduce_sum(pair), 1)
    total = l_mil + W_BOTTOMK * l_bot + W_SPARSE * l_sparse + W_SMOOTH * l_smooth
    return total, dict(mil=l_mil, bottomk=l_bot, sparse=l_sparse, smooth=l_smooth)

In [17]:
# 3.3 Train (MODE = "train") or load (MODE = "load")
def val_auc(model):
    v = table[table.split == "val"]
    return roc_auc_score(v.anomalo, [video_score(predict(model, load_features(n))) for n in v.video])

model = build_model()
if MODE == "train":
    schedule = keras.optimizers.schedules.PiecewiseConstantDecay(
        [4 * STEPS_PER_EPOCH, 8 * STEPS_PER_EPOCH], [LR, LR / 10, LR / 100])
    opt = keras.optimizers.AdamW(learning_rate=schedule, weight_decay=WEIGHT_DECAY, global_clipnorm=1.0)

    ACCUM = BATCH // MICRO_BATCH                                # micro-batches per optimizer step
    grad_acc = [tf.Variable(tf.zeros_like(v), trainable=False) for v in model.trainable_variables]

    @tf.function
    def accumulate_step(x, lengths, y):
        with tf.GradientTape() as tape:
            total, parts = compute_loss(model(x, lengths, training=True), lengths, y)
        for a, g in zip(grad_acc, tape.gradient(total, model.trainable_variables)):
            a.assign_add(g / ACCUM)
        return total, parts

    @tf.function
    def apply_step():
        opt.apply_gradients(zip([tf.identity(a) for a in grad_acc], model.trainable_variables))
        for a in grad_acc:
            a.assign(tf.zeros_like(a))

    history, it = [], iter(train_ds)
    for ep in range(EPOCHS):
        acc = defaultdict(float)
        for _ in range(STEPS_PER_EPOCH):
            for _ in range(ACCUM):
                total, parts = accumulate_step(*next(it))
                for k, v in dict(total=total, **parts).items():
                    acc[k] += float(v) / (STEPS_PER_EPOCH * ACCUM)
            apply_step()
        history.append(dict(epoch=ep + 1, **acc, val_auc=val_auc(model)))
        print(f"Epoch {ep + 1:2d} · loss {acc['total']:.4f} · val video AUC {history[-1]['val_auc']:.4f}")
    model.save_weights(WEIGHTS_PATH)
    print("Weights saved to", WEIGHTS_PATH)

    h = pd.DataFrame(history)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    h.plot(x="epoch", y=["total", "mil", "bottomk", "sparse", "smooth"], ax=ax[0], title="Training losses")
    h.plot(x="epoch", y="val_auc", ax=ax[1], title="Validation video AUC", marker="o", legend=False)
    plt.tight_layout(); plt.savefig(WORK / "figuras/entrenamiento.png", dpi=120); plt.show()
else:
    model.load_weights(WEIGHTS_PATH)
    print("Weights loaded from", WEIGHTS_PATH)

KeyboardInterrupt: 

## 4. Umbral y métricas

In [ ]:
# 4.1 Post-processing and threshold calibrated on validation
def postprocess(s, thr, fps=FPS):
    """Smoothing (forward-backward EMA) + hysteresis + merge gaps + minimum duration -> intervals in fragments."""
    def ema(v):
        out, e = np.zeros_like(v), v[0]
        for i, x in enumerate(v):
            e = EMA_ALPHA * x + (1 - EMA_ALPHA) * e; out[i] = e
        return out
    sm = (ema(s) + ema(s[::-1])[::-1]) / 2
    on, mask = False, np.zeros(len(s), bool)
    for i, x in enumerate(sm):
        on = x > (thr * OFF_RATIO if on else thr); mask[i] = on
    iv, i = [], 0
    while i < len(mask):
        if mask[i]:
            j = i
            while j < len(mask) and mask[j]:
                j += 1
            if iv and (i - iv[-1][1]) * FRAG / fps <= MERGE_GAP_SEC:
                iv[-1][1] = j
            else:
                iv.append([i, j])
            i = j
        else:
            i += 1
    iv = [(a, b) for a, b in iv if (b - a) * FRAG / fps >= MIN_EVENT_SEC]
    mask = np.zeros(len(s), bool)
    for a, b in iv:
        mask[a:b] = True
    return sm, mask, iv

VAL = table[table.split == "val"]
val_scores = {v: predict(model, load_features(v)) for v in VAL.video}
f1s = []
for thr in np.round(np.arange(0.05, 0.96, 0.01), 2):          # video-level F1: anomalous = at least one interval
    pred = np.array([len(postprocess(val_scores[v], thr)[2]) > 0 for v in VAL.video])
    true = VAL.anomalo.values.astype(bool)
    tp, fp, fn = (pred & true).sum(), (pred & ~true).sum(), (~pred & true).sum()
    f1s.append((thr, 2 * tp / max(1, 2 * tp + fp + fn)))
f1s = pd.DataFrame(f1s, columns=["threshold", "f1"])
best = f1s.threshold[f1s.f1 >= f1s.f1.max() - 1e-9]
THR = float(best.iloc[len(best) // 2])                         # middle of the best range
CONFIG_PATH.write_text(json.dumps({"threshold": THR, "weights": str(WEIGHTS_PATH)}, indent=2))
print(f"Threshold: {THR:.2f} (saved to {CONFIG_PATH.name})")

In [ ]:
# 4.2 Test metrics: AUC, F1 and IoU between detected and ground-truth anomalies
TEST = table[table.split == "test"]
test_scores = {v: predict(model, load_features(v)) for v in TEST.video}
post = {v: postprocess(test_scores[v], THR) for v in TEST.video}

def gt_mask(v, n):
    g = np.zeros(n, bool)
    for a, b in GT.get(v, []):
        g[a // FRAG: b // FRAG + 1] = True
    return g

def iou(a, b):
    return (a & b).sum() / max(1, (a | b).sum())

def iou_1d(p, q):
    inter = max(0, min(p[1], q[1]) - max(p[0], q[0]))
    return inter / max(1e-9, max(p[1], q[1]) - min(p[0], q[0]))

rows = []
for v, cls in zip(TEST.video, TEST.clase):
    s = test_scores[v]; g = gt_mask(v, len(s)); m = post[v][1]
    gts = [(a // FRAG, b // FRAG + 1) for a, b in GT.get(v, [])]
    hits = sum(any(iou_1d(p, q) >= IOU_MATCH for p in post[v][2]) for q in gts)
    rows.append(dict(video=v, clase=cls, anomalo=int(cls != "Normal"), video_score=video_score(s),
                     iou=iou(m, g) if g.any() else np.nan, n_detected=len(post[v][2]), n_gt=len(gts), hits=hits,
                     tp=(m & g).sum(), fp=(m & ~g).sum(), fn=(~m & g).sum()))
R = pd.DataFrame(rows)

S = np.concatenate([np.repeat(test_scores[v], FRAG) for v in TEST.video])
G = np.concatenate([np.repeat(gt_mask(v, len(test_scores[v])), FRAG) for v in TEST.video])
A = R.anomalo == 1
S_a = np.concatenate([np.repeat(test_scores[v], FRAG) for v in R.video[A]])
G_a = np.concatenate([np.repeat(gt_mask(v, len(test_scores[v])), FRAG) for v in R.video[A]])
tp, fp, fn = R.tp.sum(), R.fp.sum(), R.fn.sum()
METRICS = {
    "Frame AUC (standard UCF-Crime metric)": roc_auc_score(G, S),
    "Video AUC (anomalous vs normal videos)": roc_auc_score(R.anomalo, R.video_score),
    "AUC inside anomalous videos (localization)": roc_auc_score(G_a, S_a),
    "Mean IoU detected vs ground truth (anomalous videos)": R.iou[A].mean(),
    f"Events detected (IoU >= {IOU_MATCH})": R.hits.sum() / R.n_gt.sum(),
    f"Detected intervals that match an event (IoU >= {IOU_MATCH})": R.hits.sum() / max(1, R.n_detected.sum()),
    "Frame precision": tp / max(1, tp + fp),
    "Frame recall": tp / max(1, tp + fn),
    "Frame F1": 2 * tp / max(1, 2 * tp + fp + fn),
    "Normal videos with a false alarm": (R.n_detected[~A] > 0).mean(),
}
display(pd.Series(METRICS, name="test").round(4).to_frame())
(WORK / "metricas.json").write_text(json.dumps({k: float(v) for k, v in METRICS.items()}, indent=2))
R.to_csv(WORK / "resultados_test.csv", index=False)

In [ ]:
# 4.3 IoU by class and anomaly curves of some test videos
display(R[A].groupby("clase").agg(videos=("video", "size"), mean_iou=("iou", "mean"),
                                  events_detected=("hits", "sum"), events=("n_gt", "sum")).round(3))

def plot_curve(ax, name, s, sm, iv, gt=(), thr=None, fps=FPS):
    t = np.arange(len(s)) * FRAG / fps
    ax.plot(t, s, lw=0.8, c="#9DB7D5", label="score"); ax.plot(t, sm, lw=1.5, c="#1F4E79", label="smoothed")
    for k, (a, b) in enumerate(gt):
        ax.axvspan(a / fps, b / fps, color="gold", alpha=0.35, label="ground truth" if k == 0 else None)
    for k, (a, b) in enumerate(iv):
        ax.axvspan(a * FRAG / fps, b * FRAG / fps, ymax=0.06, color="red", label="detected" if k == 0 else None)
    if thr is not None:
        ax.axhline(thr, ls="--", c="gray", lw=1)
    ax.set_ylim(0, 1.03); ax.set_title(name, fontsize=9); ax.set_ylabel("anomaly")

show = R[A].sort_values("iou", ascending=False).video.tolist()
show = show[:3] + show[-3:]
fig, axs = plt.subplots(len(show), 1, figsize=(10, 2.1 * len(show)), squeeze=False)
for ax, v in zip(axs[:, 0], show):
    sm, _, iv = post[v]
    plot_curve(ax, f"{v} · IoU {R.set_index('video').loc[v, 'iou']:.2f}", test_scores[v], sm, iv, GT.get(v, []), THR)
axs[0, 0].legend(fontsize=7, ncol=4, loc="upper right"); axs[-1, 0].set_xlabel("time (s)")
plt.tight_layout(); plt.savefig(WORK / "figuras/curvas_test.png", dpi=120); plt.show()

## 5. Prueba con tus propios videos

In [ ]:
# 5.1 Video tools: CLIP feature extraction and output rendering
_CLIP = {}
def clip_features(path):
    """One CLIP ViT-B/16 feature per 16-frame fragment, with the same preprocessing as VadCLIP."""
    if not _CLIP:
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        m, _, pre = open_clip.create_model_and_transforms("ViT-B-16", pretrained="openai", device=dev)
        _CLIP.update(model=m.eval(), pre=pre, dev=dev)
    cap = cv2.VideoCapture(str(path)); frames, i = [], 0
    while True:
        ok, f = cap.read()
        if not ok:
            break
        if i % FRAG == FRAG // 2:                              # middle frame of each fragment
            rgb = cv2.cvtColor(cv2.resize(f, (340, 256)), cv2.COLOR_BGR2RGB)[16:240, 58:282]
            frames.append(_CLIP["pre"](Image.fromarray(rgb)))
        i += 1
    cap.release()
    feats = []
    with torch.no_grad():
        for j in range(0, len(frames), 128):
            feats.append(_CLIP["model"].encode_image(torch.stack(frames[j:j + 128]).to(_CLIP["dev"])).float().cpu().numpy())
    return np.concatenate(feats)

def video_features(path):
    name = Path(path).stem
    if name in FEATS:                                          # UCF-Crime video: use VadCLIP features
        return load_features(name)
    cache = WORK / "features_propias" / f"{name}.npy"
    if not cache.exists():
        np.save(cache, clip_features(path))
    return np.load(cache)

_FONT = matplotlib.font_manager.findfont("DejaVu Sans:bold")
def render(path, s, sm, mask, iv, thr, out, gt=()):
    """Banner with the score, red border during anomalies and a timeline at the bottom."""
    cap = cv2.VideoCapture(str(path))
    fps, w, h = cap.get(cv2.CAP_PROP_FPS) or FPS, int(cap.get(3)), int(cap.get(4))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); sh = max(40, h // 7)
    base = np.full((sh, w, 3), 30, np.uint8)
    for a, b in gt:
        cv2.rectangle(base, (int(a / n * w), 0), (int(b / n * w), sh), (0, 140, 180), -1)
    for a, b in iv:
        cv2.rectangle(base, (int(a / len(s) * w), sh - 7), (int(b / len(s) * w), sh), (40, 40, 230), -1)
    pts = np.array([[int((i + 0.5) / len(s) * w), int((1 - v) * (sh - 12)) + 2] for i, v in enumerate(sm)], np.int32)
    cv2.polylines(base, [pts], False, (255, 255, 255), 1, cv2.LINE_AA)
    font = ImageFont.truetype(_FONT, max(12, h // 26))
    tmp = str(out) + ".tmp.mp4"
    vw = cv2.VideoWriter(tmp, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h + sh)); f = 0
    while True:
        ok, im = cap.read()
        if not ok:
            break
        k = min(f // FRAG, len(s) - 1); anomalous = bool(mask[k])
        if anomalous:
            cv2.rectangle(im, (0, 0), (w - 1, h - 1), (40, 40, 230), max(4, h // 60))
        pil = Image.fromarray(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); dr = ImageDraw.Draw(pil)
        txt = f"{'ANOMALÍA' if anomalous else 'Normal'} · {sm[k]:.2f}".replace(".", ",")
        dr.rectangle([8, 8, 22 + dr.textlength(txt, font=font), 18 + font.size], fill=(200, 30, 30) if anomalous else (40, 40, 40))
        dr.text((15, 12), txt, font=font, fill=(255, 255, 255))
        strip = base.copy(); x = int(f / max(n, 1) * w)
        cv2.line(strip, (x, 0), (x, sh), (0, 255, 255), 2)
        vw.write(np.vstack([cv2.cvtColor(np.asarray(pil), cv2.COLOR_RGB2BGR), strip])); f += 1
    cap.release(); vw.release()
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", tmp, "-c:v", "libx264", "-pix_fmt", "yuv420p",
                    "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2", "-movflags", "+faststart", str(out)], check=True)
    os.remove(tmp)
    return out

In [ ]:
# 5.2 Choose a video and a trained model, and watch the output
VIDEO_PATH = ""                       # path to any video; empty = a UCF-Crime test video
MODEL_WEIGHTS = WEIGHTS_PATH          # any saved <name>.weights.h5 (its <name>.json holds the threshold)

if not VIDEO_PATH:
    VIDEO_PATH = next((VIDEOS[v] for v in R.sort_values("iou", ascending=False).video if v in VIDEOS), "")
if not VIDEO_PATH or not Path(VIDEO_PATH).exists():
    print("Set VIDEO_PATH to a video file.")
else:
    demo_model = build_model(); demo_model.load_weights(MODEL_WEIGHTS)
    cfg = Path(MODEL_WEIGHTS).with_name(Path(MODEL_WEIGHTS).name.replace(".weights.h5", ".json"))
    thr = json.loads(cfg.read_text())["threshold"] if cfg.exists() else THR
    name = Path(VIDEO_PATH).stem
    fps = cv2.VideoCapture(str(VIDEO_PATH)).get(cv2.CAP_PROP_FPS) or FPS
    s = predict(demo_model, video_features(VIDEO_PATH))
    sm, mask, iv = postprocess(s, thr, fps)
    gt = GT.get(name, [])
    out = render(VIDEO_PATH, s, sm, mask, iv, thr, WORK / "demo" / f"{name}_salida.mp4", gt)

    print(f"Video: {name} · video score {video_score(s):.2f} · threshold {thr:.2f}")
    events = pd.DataFrame([dict(start_s=round(a * FRAG / fps, 2), end_s=round(b * FRAG / fps, 2),
                                max_score=round(float(s[a:b].max()), 3)) for a, b in iv], columns=["start_s", "end_s", "max_score"])
    print("Anomalous segments:" if len(events) else "No anomaly detected.")
    display(events)
    if gt:
        print(f"IoU with the ground truth: {iou(mask, gt_mask(name, len(s))):.3f}")
    fig, ax = plt.subplots(figsize=(10, 2.4))
    plot_curve(ax, name, s, sm, iv, gt, thr, fps); ax.set_xlabel("time (s)"); ax.legend(fontsize=7, ncol=4)
    plt.tight_layout(); plt.show()
    display(Video(str(out), embed=True, width=720))